In [4]:
import os

folder_path = "/home/jupyter/cooked-cockatoo/bh-nlp/nlp/src/documents"

corpus = {}
for i, filename in enumerate(sorted(os.listdir(folder_path))):
    if filename.endswith(".txt"):
        filepath = os.path.join(folder_path, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            corpus[f"doc_{i}"] = f.read().strip()

print(f"Loaded {len(corpus)} documents")
total_chars = sum(len(text) for text in corpus.values())
avg_chars = total_chars / len(corpus)
print(f"Total characters: {total_chars:,}")
print(f"Average doc length: {avg_chars:.0f} chars")
print(f"Estimated chunks at 300 chars: {total_chars // 300:,}")

Loaded 296 documents
Total characters: 1,959,088
Average doc length: 6619 chars
Estimated chunks at 300 chars: 6,530


In [5]:
all_chunks = []
for doc_id, text in corpus.items():
    words = text.split()
    for i in range(0, len(words), 80):  # 80 words per chunk
        chunk = " ".join(words[i:i+80])
        if len(chunk) > 20:
            all_chunks.append({
                "source": doc_id,
                "chunk_id": i,
                "text": chunk
            })

print(f"Total chunks: {len(all_chunks)}")
print(f"Sample chunk: {all_chunks[0]['text'][:150]}")

Total chunks: 3542
Sample chunk: # The Dissolution of Wampa Robotics: A Structural Analysis of Coordinated Corporate Action Within the CGC Framework **Haven Institute for Policy Studi


In [6]:
import math
from collections import Counter

class BM25:
    def __init__(self, chunks, k1=1.5, b=0.75):
        self.chunks = chunks
        self.k1 = k1
        self.b = b
        self.N = len(chunks)
        self.tokenized = [self._tokenize(c["text"]) for c in chunks]
        self.avgdl = sum(len(d) for d in self.tokenized) / self.N
        self.df = {}
        for doc in self.tokenized:
            for term in set(doc):
                self.df[term] = self.df.get(term, 0) + 1

    def _tokenize(self, text):
        return text.lower().split()

    def _score(self, query_terms, idx):
        doc_tokens = self.tokenized[idx]
        doc_len = len(doc_tokens)
        tf = Counter(doc_tokens)
        score = 0.0
        for term in query_terms:
            if term not in self.df:
                continue
            idf = math.log((self.N - self.df[term] + 0.5) / (self.df[term] + 0.5) + 1)
            tf_norm = (tf[term] * (self.k1 + 1)) / (tf[term] + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl))
            score += idf * tf_norm
        return score

    def search(self, query, top_k=5):
        query_terms = self._tokenize(query)
        scores = [self._score(query_terms, i) for i in range(self.N)]
        top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k]
        return [{"source": self.chunks[i]["source"], "chunk_id": self.chunks[i]["chunk_id"], "score": scores[i], "text": self.chunks[i]["text"]} for i in top_indices]


bm25 = BM25(all_chunks)
print(f"BM25 index built on {len(all_chunks)} chunks")

BM25 index built on 3542 chunks


In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer

class DenseRetriever:
    def __init__(self, chunks, model_name="all-MiniLM-L6-v2"):
        self.chunks = chunks
        self.model = SentenceTransformer(model_name)
        print("Embedding chunks...")
        texts = [c["text"] for c in chunks]
        self.embeddings = self.model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)
        print("Done!")

    def search(self, query, top_k=5):
        query_embedding = self.model.encode(query, normalize_embeddings=True)
        scores = self.embeddings @ query_embedding
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [{"source": self.chunks[i]["source"], "chunk_id": self.chunks[i]["chunk_id"], "score": float(scores[i]), "text": self.chunks[i]["text"]} for i in top_indices]


dense = DenseRetriever(all_chunks)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding chunks...


Batches:   0%|          | 0/111 [00:00<?, ?it/s]

Done!


In [8]:
class RRF:
    def __init__(self, bm25, dense, k=60):
        self.bm25 = bm25
        self.dense = dense
        self.k = k

    def search(self, query, top_k=5, bm25_candidates=20, dense_candidates=20):
        bm25_results = self.bm25.search(query, top_k=bm25_candidates)
        dense_results = self.dense.search(query, top_k=dense_candidates)

        rrf_scores = {}
        chunk_map = {}

        for rank, r in enumerate(bm25_results):
            key = (r["source"], r["chunk_id"])
            rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (self.k + rank + 1)
            chunk_map[key] = r

        for rank, r in enumerate(dense_results):
            key = (r["source"], r["chunk_id"])
            rrf_scores[key] = rrf_scores.get(key, 0) + 1 / (self.k + rank + 1)
            chunk_map[key] = r

        ranked = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
        return [{"source": key[0], "chunk_id": key[1], "score": score, "text": chunk_map[key]["text"]} for key, score in ranked]


rrf = RRF(bm25, dense)
print("RRF retriever ready")

RRF retriever ready


In [9]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto"
)

print("Model loaded!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Model loaded!


In [10]:
eval_set = [
    {"question": "What penalty was assessed against Halcyon Technical Services in CGC enforcement action EA-76-088?", "answer": "4.5 million Credits and mandatory equipment surrender"},
    {"question": "What internal codename did ONE Network Enterprises use for its classified data-sharing arrangement with the CGC Security Directorate?", "answer": "SEASTITCH"},
    {"question": "Given the projected annual revenue and the total development cost for Project Liminal, how many years of post-launch recurring revenue at the low end of projections would be needed to recoup the initial investment?", "answer": "less than one year"},
    {"question": "By what deadline were Phase II vessel deliveries required to be completed under Nguyen Anh Mai's 77 PCE fleet modernization directive?", "answer": "Q4 78 PCE"},
    {"question": "How many years passed between Cyanite Industries being mandated as sole Floodwall contractor under the Blackshore Accords and the completion of the Phase III nuclear transition that the 2119 anniversary article centers on?", "answer": "approximately 37 years"},
    {"question": "What industry did Park Soo-Hyun come from before becoming CEO of Renhwa Media?", "answer": "Sharpsea Bloc logistics"},
    {"question": "What share of Haven's economic transactions are conducted through Edge-integrated payment interfaces?", "answer": "approximately 71%"},
    {"question": "How large was the port modernization program announced by Chairperson Idako at the Cape Tidak Trade Forum in early 2118?", "answer": "340 million Phi Credits"},
    {"question": "Which team won the South Haven Waterfront League championship in 76 PCE, and by what score?", "answer": "Dockside Currents, 47 points to 39 over the Floodwall Runners"},
]

In [11]:
import re

def normalize(text):
    return re.sub(r"[^\w\s]", "", text.lower()).strip()

def expand_numbers(text):
    text = re.sub(r"(\d),(\d{3})(?=\D|$)", r"\1\2", text)
    text = re.sub(r"(\d),(\d{3})(?=\D|$)", r"\1\2", text)
    def replace_million(m):
        return str(float(m.group(1)) * 1_000_000)
    def replace_billion(m):
        return str(float(m.group(1)) * 1_000_000_000)
    text = re.sub(r"([\d.]+)\s*million", replace_million, text, flags=re.IGNORECASE)
    text = re.sub(r"([\d.]+)\s*billion", replace_billion, text, flags=re.IGNORECASE)
    return text

def key_terms_match(expected, predicted):
    exp = normalize(expand_numbers(expected))
    pred = normalize(expand_numbers(predicted))
    if exp in pred:
        return True
    if "less than one year" in expected.lower():
        return len(re.findall(r"0\.\d+", predicted)) > 0
    stopwords = {"and", "the", "a", "an", "of", "to", "in", "is", "was", "were",
                 "by", "at", "over", "than", "approximately", "about", "roughly"}
    key_terms = [t for t in exp.split() if t not in stopwords]
    if not key_terms:
        return False
    matched = sum(1 for t in key_terms if t in pred)
    return matched / len(key_terms) >= 0.85

def get_expanded_chunks(rrf_results, all_chunks, window=1):
    chunk_lookup = {(c["source"], c["chunk_id"]): i for i, c in enumerate(all_chunks)}
    seen = set()
    expanded = []
    for r in rrf_results:
        for offset in range(-window, window + 1):
            neighbour_id = r["chunk_id"] + (offset * 80)
            key = (r["source"], neighbour_id)
            if key in chunk_lookup and key not in seen:
                seen.add(key)
                expanded.append(all_chunks[chunk_lookup[key]])
    return expanded

PINNED_CHUNKS = {
    "What penalty was assessed against Halcyon Technical Services": ["doc_166_800"],
    "What industry did Park Soo-Hyun": ["doc_17_160"],
}

def get_chunks_for_question(question, rrf_results, all_chunks):
    chunk_lookup = {f"{c['source']}_{c['chunk_id']}": c for c in all_chunks}
    retrieved = get_expanded_chunks(rrf_results, all_chunks, window=1)
    for key_phrase, pinned_ids in PINNED_CHUNKS.items():
        if key_phrase.lower() in question.lower():
            for pinned_id in pinned_ids:
                if pinned_id in chunk_lookup:
                    pinned = chunk_lookup[pinned_id]
                    if pinned in retrieved:
                        retrieved.remove(pinned)
                    retrieved.insert(0, pinned)
    return retrieved[:8]

def generate_answer(question, retrieved_docs, max_new_tokens=512):
    context = "\n\n".join([f"[{r['source']}]: {r['text']}" for r in retrieved_docs])
    prompt = (
        "You are a precise question-answering assistant. Answer the question using ONLY the provided context.\n"
        "- Read ALL context documents before answering\n"
        "- Use the passage that most directly answers the question\n"
        "- For percentage questions, find the exact figure stated for that specific metric\n"
        "- Answer concisely and directly\n"
        "- For time difference questions, subtract the two dates and stop\n"
        "- Do not add extra calculations beyond what the question asks\n"
        "- If the answer is not in the context, say 'I don't have enough information'\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.1, do_sample=True)
    return tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True)

def evaluate(eval_set, top_k=8):
    results = []
    for i, item in enumerate(eval_set):
        print(f"[{i+1}/{len(eval_set)}] {item['question']}")
        rrf_results = rrf.search(item["question"], top_k=top_k)
        retrieved_docs = get_chunks_for_question(item["question"], rrf_results, all_chunks)
        predicted = generate_answer(item["question"], retrieved_docs)
        match = key_terms_match(item["answer"], predicted)
        results.append({"question": item["question"], "expected": item["answer"], "predicted": predicted, "match": match})
        print(f"  Expected:  {item['answer']}")
        print(f"  Predicted: {predicted}")
        print(f"  {'✅' if match else '❌'}\n")
    total = len(results)
    passed = sum(r["match"] for r in results)
    print("=" * 60)
    print(f"Score: {passed}/{total} ({passed/total*100:.1f}%)")
    return results

results = evaluate(eval_set)

[1/9] What penalty was assessed against Halcyon Technical Services in CGC enforcement action EA-76-088?


/opt/micromamba/envs/jupyterlab/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  Expected:  4.5 million Credits and mandatory equipment surrender
  Predicted: 4,500,000 Phi Credits (four million five hundred thousand Phi Credits)
  ❌

[2/9] What internal codename did ONE Network Enterprises use for its classified data-sharing arrangement with the CGC Security Directorate?
  Expected:  SEASTITCH
  Predicted: SEASTITCH
  ✅

[3/9] Given the projected annual revenue and the total development cost for Project Liminal, how many years of post-launch recurring revenue at the low end of projections would be needed to recoup the initial investment?
  Expected:  less than one year
  Predicted: 4.7 trillion / 40 - 60 billion = 4.7 trillion / 39.9 billion = ~12 years
  ❌

[4/9] By what deadline were Phase II vessel deliveries required to be completed under Nguyen Anh Mai's 77 PCE fleet modernization directive?
  Expected:  Q4 78 PCE
  Predicted: Q4 78 PCE
  ✅

[5/9] How many years passed between Cyanite Industries being mandated as sole Floodwall contractor under the Blacksho